In [ ]:
import sys
import os
import json
import pandas as pd

In [ ]:
if 'google.colab' in sys.modules: 
    if not os.path.exists('/content/nlp_course_project'):
        !git clone https://github.com/Karoshi-man/nlp_course_project.git
    
    %cd /content/nlp_course_project
    sys.path.append('/content/nlp_course_project')
    
    FOLDER_ID = '1pIDpBFJ33L9XrldgXEXiAnLRNCs6f0gb'
    
    os.makedirs('/content/nlp_course_project/data', exist_ok=True)
    !gdown --folder https://drive.google.com/drive/folders/{FOLDER_ID} -O /content/nlp_course_project/data/
    
    data_dir = '/content/nlp_course_project/data'

else:
    sys.path.append(os.path.abspath('..'))
    data_dir = '../data'

In [ ]:
gold_path = os.path.join(data_dir, "sample", "lab4_gold_ie.jsonl")

gold_data = []
with open(gold_path, 'r', encoding='utf-8') as f:
    for line in f:
        gold_data.append(json.loads(line))

In [ ]:
from src.ie_rules import extract_all, ENGLISH_DICT
print(list(ENGLISH_DICT.keys())[:5], "...")

In [ ]:
sample_text = gold_data[0]["text"]
extracted = extract_all(sample_text)

print(f"Текст: {sample_text}\n")
print("Знайдені сутності:")
print(json.dumps(extracted, indent=2, ensure_ascii=False))

In [ ]:
metrics = {
    "SALARY": {"TP": 0, "FP": 0},
    "EXPERIENCE_YEARS": {"TP": 0, "FP": 0},
    "ENGLISH_LEVEL": {"TP": 0, "FP": 0}
}

for case in gold_data:
    text = case["text"]
    gold_entities = case["gold"]
    predicted_entities = extract_all(text)
    pred_flat = []
    for p in predicted_entities:
        val = p["value"]["min"] if p["field_type"] == "SALARY" else p["value"]
        pred_flat.append({"type": p["field_type"], "value": val})
        
    gold_flat = [{"type": g["type"], "value": g["value"]} for g in gold_entities]
    
    for pred in pred_flat:
        match = next((g for g in gold_flat if g["type"] == pred["type"] and g["value"] == pred["value"]), None)
        if match:
            metrics[pred["type"]]["TP"] += 1
            gold_flat.remove(match)
        else:
            metrics[pred["type"]]["FP"] += 1

In [ ]:
print(f"{'Поле':<20} | {'Precision':<10} | {'TP':<4} | {'FP':<4}")
for field, counts in metrics.items():
    tp, fp = counts["TP"], counts["FP"]
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    print(f"{field:<20} | {precision:<10.2f} | {tp:<4} | {fp:<4}")

### 6. Error analysis (10 FP прикладів)

У процесі розробки Rule-based модуля ми виявили та виправили наступні False Positives, додавши відповідні анти-правила:

1. **Проблема:** "Кандидати від 18 років"
   * **Що спрацювало:** `EXPERIENCE_YEARS` = 18.0
   * **Чому помилка:** Патерн витягнув вік замість досвіду.
   * **Анти-правило:** Евристика `if exp_val > 15: continue`.

2. **Проблема:** "Досвід у B2B продажах"
   * **Що спрацювало:** `ENGLISH_LEVEL` = B2
   * **Чому помилка:** Знайдено частину слова.
   * **Анти-правило:** Додано межі слів `\b` навколо ключів словника.

3. **Проблема:** "Компанія заснована в 2000 році"
   * **Що спрацювало:** `SALARY` = 2000
   * **Чому помилка:** Число без контексту валюти.
   * **Анти-правило:** Обов'язкова вимога знаку валюти поруч у regex.

4. **Проблема:** "Знання Python 3"
   * **Що спрацювало:** `EXPERIENCE_YEARS` = 3.0
   * **Чому помилка:** Збіг числа біля технології.
   * **Анти-правило:** Вимога контекстних слів (роки, years) у regex.

5. **Проблема:** "Зарплата від 3k до 4.5k USD"
   * **Що спрацювало:** `SALARY min` = 5000 (витягло ".5k")
   * **Чому помилка:** Патерн не підтримував крапку перед "k".
   * **Анти-правило:** Додано `(?:[.,]\d{1,2})?` у регулярку та обробку float.

6. **Проблема:** "Рейт 40$ / година"
   * **Що спрацювало:** `SALARY` = 40
   * **Чому помилка:** Погодинна оплата замість місячної.
   * **Анти-правило:** Евристичний мінімум `if val_min < 200: continue`.

7. **Проблема:** "Володіння C++"
   * **Що спрацювало:** `ENGLISH_LEVEL` = C1
   * **Чому помилка:** Збіг літери С.
   * **Анти-правило:** Жорсткий мапінг лише за повними ключами словника.

8. **Проблема:** "Бонус 20%"
   * **Що спрацювало:** `SALARY` = 20
   * **Чому помилка:** Відсотки сплутано з сумою.
   * **Анти-правило:** Ігнорування чисел зі знаком %.

9. **Проблема:** "Upper-Intermediate"
   * **Що спрацювало:** `ENGLISH_LEVEL` = B1 (витягло Intermediate)
   * **Чому помилка:** Пошук знайшов коротше слово першим.
   * **Анти-правило:** Сортування ключів словника за довжиною `reverse=True`.

10. **Проблема:** "Бюджет 120 000 грн"
    * **Що спрацювало:** `SALARY` = 0
    * **Чому помилка:** Пробіл розірвав число.
    * **Анти-правило:** Додано підтримку пробілів `(?:[ .,]\d{3})*` у regex.

In [ ]:
summary_path = os.path.join(".." if not 'google.colab' in sys.modules else "", "docs", "audit_summary_lab4.md")

markdown_content = """# Audit Summary Lab 4: Rule-based Information Extraction

## 1. Загальна інформація
* **Метод:** Rule-based (Regex + Dictionaries)
* **Поля для екстракції:** `SALARY`, `EXPERIENCE_YEARS`, `ENGLISH_LEVEL`

## 2. Метрики якості (Gold Subset)
| Field Type | Precision | True Positives | False Positives |
| :--- | :---: | :---: | :---: |
| **SALARY** | 1.00 | 4 | 0 |
| **EXPERIENCE_YEARS** | 1.00 | 7 | 0 |
| **ENGLISH_LEVEL** | 1.00 | 7 | 0 |

## 3. Error Analysis
Алгоритм успішно обробляє edge cases (B2B, вік 18 років, 4.5k зарплата) завдяки впровадженим евристикам та регулярним виразам із негативним lookbehind.
"""

os.makedirs(os.path.dirname(summary_path), exist_ok=True)
with open(summary_path, "w", encoding="utf-8") as f:
    f.write(markdown_content)